[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/your_notebook.ipynb)

In [ ]:
#install (run once):


!pip install -q transformers datasets peft scikit-learn accelerate


In [ ]:

#Step 1: Prepare your dataset


import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "hf://datasets/PolyAI/banking77@refs/convert/parquet/default/train/0000.parquet",
        "test":  "hf://datasets/PolyAI/banking77@refs/convert/parquet/default/test/0000.parquet",
    },
)

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

# Banking77 has only train + test; carve a validation split out of train.
train_data, val_data = train_test_split(
    train_df, test_size=0.15, random_state=42, stratify=train_df["label"]
)
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_df.reset_index(drop=True)

num_labels = int(pd.concat([train_df["label"], test_df["label"]]).max()) + 1

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")
print(f"Number of labels: {num_labels}")


In [ ]:

#Step 2: Apply LoRA to the model


from transformers import AutoTokenizer, BertForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

for name, module in model.named_modules():
    print(name)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["query", "value"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding=False, max_length=128)

train_ds = Dataset.from_pandas(train_data).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_data).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_data).map(tokenize, batched=True)

keep = ["input_ids", "attention_mask", "label"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep])
val_ds   = val_ds.remove_columns([c for c in val_ds.column_names   if c not in keep])
test_ds  = test_ds.remove_columns([c for c in test_ds.column_names  if c not in keep])


In [ ]:

#Step 3: Fine-tune the model with LoRA


from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall":    recall_score(labels, preds, average="weighted", zero_division=0),
        "f1":        f1_score(labels, preds, average="weighted", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    learning_rate=2e-4,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
#Step 4: Evaluate the LoRA-fine-tuned model


results = trainer.evaluate(eval_dataset=test_ds)

pred_output = trainer.predict(test_ds)
predictions = pred_output.predictions.argmax(-1)
true_labels = np.array(test_ds["label"])

accuracy  = results.get("eval_accuracy", accuracy_score(true_labels, predictions))
precision = precision_score(true_labels, predictions, average="weighted", zero_division=0)
recall    = recall_score(true_labels, predictions, average="weighted", zero_division=0)
f1        = f1_score(true_labels, predictions, average="weighted", zero_division=0)

print(f"Test Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")


In [ ]:

#Step 5: Optimize LoRA for your task


alpha = 16
dropout_rate = 0.1
use_bias = "none"  # PEFT accepts "none", "all", or "lora_only"

base_model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
new_lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=4,  # experiment: try 2, 4, 8, 16
    lora_alpha=alpha,
    lora_dropout=dropout_rate,
    bias=use_bias,
    target_modules=["query", "value"],
)
model = get_peft_model(base_model, new_lora_config)
model.print_trainable_parameters()

print(f"Rank: {new_lora_config.r}")
print(f"Alpha: {alpha}")
print(f"Dropout Rate: {dropout_rate}")
print(f"Bias mode: {use_bias}")